# Exercise 02 — Flickr API

## What are API's ?

![API](https://media.geeksforgeeks.org/wp-content/uploads/20230216170349/What-is-an-API.png)

An API (Application Programming Interface) is a set of rules that allows different software programs to talk to each other.

You can think of it like a menu in a restaurant: the menu lists the dishes you can order, along with a description. When you tell the waiter what you want, the kitchen (the system behind the API) prepares it and sends it back to you.

In the same way, an API defines what requests you can make and what responses you’ll get—for example, asking a weather API for today’s temperature or a Flickr API for recent photos.

APIs are used everywhere: they let your app get data from a website, send messages, control devices, or connect services together.

## What is Flickr?

![Flickr](https://www.flickrhelp.com/hc/article_attachments/4419907628308/unnamed__1_.png)  

Flickr is an online platform for sharing, storing, and discovering photos and videos. It was one of the first major social networks built around photography.

Users can:

- Upload their own images and organize them into albums.
- Add tags, titles, and descriptions to make photos searchable.
- Explore images from other users around the world.
- Comment, like, and join photography groups or communities.

Flickr is especially popular among photographers, artists, and researchers because it provides high-quality image hosting and rich metadata (like geolocation, upload time, and camera info).

A typical Flickr post can be viewed here: [link](https://www.flickr.com/photos/92959567@N00/227080829).  

It also offers a simple public API, which allows developers to access that photo data programmatically — for example, to download geotagged images, analyze tags, or visualize where and what people photograph.

The objective of this exercise is to create a table containing images from a specific area along with their associated information.  

To achieve this, we will use the "Python Flickr API" library to handle authentication. Once authenticated, you can utilize the methods detailed in the API documentation [here](https://www.flickr.com/services/api/).  

The API methods fall under different authentication levels, such as:  
- *Requires 'write' permission for authentication.*  
- *Requires 'read' permission for authentication.*  
- *Does not require authentication.*  

Focus only on the methods that do not require authentication and disregard those needing "write" or "read" permissions.



## Geo Search

### Flickr subscription and API key

To run the live Flickr searches in this tutorial, **each student needs a Flickr Pro subscription and their own API key**. Flickr currently restricts requests for new API keys to Pro subscribers. Sign in or create a Flickr account, subscribe to Pro, then visit the [Flickr API Keys page](https://www.flickr.com/services/api/keys/) to apply for a non-commercial API key for educational use. Applications may require review, so arrange this before the workshop. Keep your API key and API secret private: never publish them on GitHub or include them in submitted notebooks. See [Flickr's official API guidance](https://www.flickrhelp.com/hc/en-us/articles/4404070036884-Flickr-API).

### Preparation & First Searches

Install the Python libraries if they are not already available. In Jupyter or Colab, run `%pip install flickrapi pandas folium` in a separate code cell. You only need to install them once per environment.

**Entering your Flickr credentials:** Each API application has a **key** and a **secret**. The next cell prompts you to enter both. The `getpass()` function hides what you type, so neither credential needs to appear in the notebook's code or printed output. The values stay in memory while the notebook kernel is running. If you restart the kernel, you will need to enter them again.

Do not replace these prompts with your actual key or secret when saving or sharing the notebook.

In [2]:
from getpass import getpass

api_key = getpass("Flickr API key: ")
api_secret = getpass("Flickr API secret: ")

Import the libraries we need: `flickrapi` to communicate with Flickr, `pandas` for tables, and `folium` to draw interactive maps. The `FlickrAPI()` object uses the key and secret entered above to connect to Flickr.

In [3]:
import flickrapi
import pandas as pd
import folium


#connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format='parsed-json')


### Define search location and display map

![Instructional diagram: bounding box](assets/bounding-box.svg)

Flickr lets us search for photographs taken within a geographical area. We define this area with a **bounding box**: a rectangle described by its northern, southern, western and eastern limits.

The coordinates below describe a small area in central London. Latitude measures position north or south; longitude measures position east or west. You can find coordinates for another location using [OpenStreetMap](https://www.openstreetmap.org/).

There are **two different coordinate orders** to remember:

| Where we use coordinates | Required order |
| --- | --- |
| **Folium**, for map positions | `[latitude, longitude]` |
| **Flickr**, for a bounding box | `west longitude, south latitude, east longitude, north latitude` |

First we draw the search rectangle and its centre point on a Folium map. This lets us check the area visually before requesting any photographs.

In [6]:
# Centre of our search area, in latitude / longitude order
centre_latitude = 51.51235
centre_longitude = -0.11720

# Bounding box: the four limits of our search rectangle
lat_north = 51.51491
lat_south = 51.50893
long_west = -0.12167
long_east = -0.11171

# Start with just two photos per page so the response is easy to inspect
per_page = 2

# Display the search area on a map
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    #tiles="OpenStreetMap",
    zoom_start=16,
    control_scale=True
)

# Mark the centre
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6
).add_to(map_london)

# Folium rectangle corners use [latitude, longitude] order
folium.Rectangle(
    bounds=[[lat_south, long_west], [lat_north, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue"
).add_to(map_london)

map_london

### Search Flickr for photographs

The main API method is [`flickr.photos.search`](https://www.flickr.com/services/api/flickr.photos.search.html). We send it the area we want to search and ask Flickr to return photographs with geographical information.

| Argument | Purpose |
| --- | --- |
| `bbox` | Search rectangle: **west, south, east, north** |
| `per_page` | Maximum number of photographs returned on each results page |
| `page` | Which results page to retrieve, starting at 1 |
| `has_geo=1` | Require photographs with geographical information |
| `extras` | Request additional details such as coordinates, tags and dates |

Unlike Folium, Flickr expects the bounding box as a **comma-separated string**, with longitude first. We construct it from the four variables rather than typing the coordinates again. By requesting only two photographs initially, the returned data is easier to inspect.

In [7]:
# Flickr bbox order: west longitude, south latitude, east longitude, north latitude
bbox = f"{long_west},{lat_south},{long_east},{lat_north}"

photos = flickr.photos.search(
    bbox=bbox,
    per_page=per_page,
    page=1,
    has_geo=1,
    extras="geo,description,tags,views,media,url_n,date_taken,owner_name"
)

print(photos)

{'photos': {'page': 1, 'pages': 38095, 'perpage': 2, 'total': 76190, 'photo': [{'id': '55660443740', 'owner': '47072881@N06', 'secret': '8c3e08c57e', 'server': '65535', 'farm': 66, 'title': 'Exeter Street - 33136', 'ispublic': 1, 'isfriend': 0, 'isfamily': 0, 'description': {'_content': 'Charles Octavius Parnell (1864)\n\nCorner detail. Built as the Victoria Sporting Club.'}, 'datetaken': '2022-10-24 10:00:00', 'datetakengranularity': 0, 'datetakenunknown': '0', 'ownername': 'padraic collins', 'views': '26', 'tags': 'architecture orthographic exeterstreet historicennglandlistentrynumber1357028 charlesoctaviusparnell victoriasportingclub clockbootsbuzz london coventgarden', 'latitude': '51.511947', 'longitude': '-0.119745', 'accuracy': '16', 'context': 0, 'place_id': None, 'woeid': '2652222', 'geo_is_public': 1, 'geo_is_contact': 0, 'geo_is_friend': 0, 'geo_is_family': 0, 'media': 'photo', 'media_status': 'ready', 'url_n': 'https://live.staticflickr.com/65535/55660443740_8c3e08c57e_n.jp

### Parsing the results: JSON

![Instructional diagram: json structure](assets/json-structure.svg)

The search result is stored in the variable `photos`. Flickr sends its response in **JSON** (JavaScript Object Notation), a widely used format for exchanging structured data through APIs. Because we connected to Flickr with `format="parsed-json"`, the `flickrapi` library has already converted that JSON response into familiar Python **dictionaries** and **lists**.

JSON objects use named keys, much like Python dictionaries. Arrays contain sequences of values, much like Python lists. They can be nested inside one another.

For example, this is a simple JSON object containing a list:

```json
{
  "city": "London",
  "locations": ["Westminster", "Soho", "Bloomsbury"],
  "has_photos": true,
  "note": null
}
```

Notice that JSON writes `true`, `false` and `null` in lowercase. Once converted into Python, they become `True`, `False` and `None`.

### Inspect the Flickr response

The response contains an outer dictionary, a nested dictionary under the key `"photos"`, and a list of individual photograph records under `"photo"`. Each photograph is another dictionary, with keys such as `"id"`, `"title"` and `"latitude"`.

The structure is roughly:

```text
photos                         Python dictionary
└── "photos"                  Dictionary with search results
    ├── "page"                Current page number
    ├── "pages"               Total number of pages
    └── "photo"               List of photograph dictionaries
        ├── [0]               First photograph
        │   ├── "id"          Flickr photograph ID
        │   └── "title"       Photograph title
        └── [1]               Second photograph
```

We can use Python's built-in `pprint` module (*pretty print*) to display the response with indentation. This is useful when investigating an unfamiliar API.

In [9]:
import pprint

pprint.pprint(photos, compact=True)

{'photos': {'page': 1,
            'pages': 38095,
            'perpage': 2,
            'photo': [{'accuracy': '16',
                       'context': 0,
                       'datetaken': '2022-10-24 10:00:00',
                       'datetakengranularity': 0,
                       'datetakenunknown': '0',
                       'description': {'_content': 'Charles Octavius Parnell '
                                                   '(1864)\n'
                                                   '\n'
                                                   'Corner detail. Built as '
                                                   'the Victoria Sporting '
                                                   'Club.'},
                       'farm': 66,
                       'geo_is_contact': 0,
                       'geo_is_family': 0,
                       'geo_is_friend': 0,
                       'geo_is_public': 1,
                       'height_n': 232,
                       'id':

### Access values in nested dictionaries and lists

We can access each level using square brackets. **Strings** select dictionary keys, while **integers** select positions in a list. Python list positions start at zero.

Begin with the outer dictionary and explore the nested result:

In [10]:
print(photos.keys())
print(photos["photos"]["page"])
print(photos["photos"]["pages"])

dict_keys(['photos', 'stat'])
1
38095


The list of photographs is found at `photos["photos"]["photo"]`. Let us store it in a shorter variable so the expressions become easier to read.

In [11]:
photo_list = photos["photos"]["photo"]

print("Photographs on this page:", len(photo_list))
pprint.pprint(photo_list, compact=True)

Photographs on this page: 2
[{'accuracy': '16',
  'context': 0,
  'datetaken': '2022-10-24 10:00:00',
  'datetakengranularity': 0,
  'datetakenunknown': '0',
  'description': {'_content': 'Charles Octavius Parnell (1864)\n'
                              '\n'
                              'Corner detail. Built as the Victoria Sporting '
                              'Club.'},
  'farm': 66,
  'geo_is_contact': 0,
  'geo_is_family': 0,
  'geo_is_friend': 0,
  'geo_is_public': 1,
  'height_n': 232,
  'id': '55660443740',
  'isfamily': 0,
  'isfriend': 0,
  'ispublic': 1,
  'latitude': '51.511947',
  'longitude': '-0.119745',
  'media': 'photo',
  'media_status': 'ready',
  'owner': '47072881@N06',
  'ownername': 'padraic collins',
  'place_id': None,
  'secret': '8c3e08c57e',
  'server': '65535',
  'tags': 'architecture orthographic exeterstreet '
          'historicennglandlistentrynumber1357028 charlesoctaviusparnell '
          'victoriasportingclub clockbootsbuzz london coventgarden',


If Flickr returned at least one photograph, `photo_list[0]` selects the **first photograph dictionary**, and `["id"]` extracts its ID. We check that the list is not empty before accessing its first element.

In [12]:
if photo_list:
    first_photo = photo_list[0]
    print("Photo ID:", first_photo["id"])
    print("Title:", first_photo.get("title", "No title"))
else:
    print("No photographs were returned for this search.")

Photo ID: 55660443740
Title: Exeter Street - 33136


Here `.get("title", "No title")` retrieves the title if it exists, and supplies a fallback if it does not. By contrast, accessing a missing key directly with square brackets raises a `KeyError`.

The important idea is that **an API response is a structured collection of values**, not yet a spreadsheet-like table. In the next section we will retrieve several pages and convert the individual photo dictionaries into a Pandas DataFrame.

### Getting all pages (Pagination)

![Instructional diagram: pagination](assets/pagination.svg)

Our first request returned just **two photographs**, because we set `per_page = 2`. Flickr may have many more matching photographs, but it divides the response into **pages**. This is called *pagination*.

The response we already saved as `photos` tells us how many matching photographs and result pages are available:

- `photos["photos"]["total"]`: total number of matching photographs.
- `photos["photos"]["pages"]`: number of pages of results.
- `photos["photos"]["page"]`: the page returned by our request.

We can request another page by changing the `page` argument. Later we will use a loop to retrieve several pages and combine their photographs into a table.

### Determine the number of pages

We can read the counts directly from our first API response; we do not need to make the same request again.

In [13]:
# Use the response from our first search
total_pages = int(photos["photos"]["pages"])
total_photos = int(photos["photos"]["total"])

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")
print(f"Current page: {photos['photos']['page']}")

# Display one example if the search returned photographs
if photos["photos"]["photo"]:
    first_photo = photos["photos"]["photo"][0]
    print("First photo ID:", first_photo["id"])
    print("Title:", first_photo.get("title", "No title"))
else:
    print("No photographs found in this area.")

Total photos: 76190
Total pages: 38095
Current page: 1
First photo ID: 55660443740
Title: Exeter Street - 33136


### Extract Page Information

We now define a function called `get_page()`. It requests one results page, loops through its photograph records, and converts them into a **Pandas DataFrame**.

Each Flickr photograph is a dictionary. We extract the fields we want into a new dictionary (`new_row`), then collect these dictionaries in a list called `rows`. Finally, `pd.DataFrame(rows)` turns the list into a table.

Some photographs may not include every optional field, particularly image URLs or descriptions. `.get()` lets us supply a fallback value instead of stopping with a `KeyError`.

The function retains the column names used by the map below. For now, we keep the photo ID and geographical coordinates alongside the descriptive information.

In [14]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )

    photos_on_page = response["photos"]["photo"]
    rows = []

    for photo in photos_on_page:
        description = photo.get("description") or {}
        if isinstance(description, dict):
            description = description.get("_content", "")

        new_row = {
            "id": photo.get("id"),
            "server": photo.get("server"),
            "secret": photo.get("secret"),
            "title": photo.get("title", ""),
            "tags": photo.get("tags", ""),
            "views": photo.get("views"),
            "description": description,
            "date_taken": photo.get("datetaken"),
            "latitude": photo.get("latitude"),
            "longitude": photo.get("longitude"),
            "url_s": photo.get("url_s"),
            "owner_name": photo.get("ownername"),
            "owner": photo.get("owner"),
            "media": photo.get("media"),
        }
        rows.append(new_row)

    columns = [
        "id", "server", "secret", "title", "tags", "views",
        "description", "date_taken", "latitude", "longitude",
        "url_s", "owner_name", "owner", "media",
    ]
    return pd.DataFrame(rows, columns=columns)


# Try the function on the first page
get_page(bbox, 1, per_page)

,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,55658638619,65535,761c1ecf62,"London 2026, Sir John Soane's Museum,",,3,,2026-10-11 18:33:56,51.514488,-0.118998,https://live.staticflickr.com/65535/5565863861...,The Island Kings,15775662@N00,photo
1,55658438656,65535,4fe0b610d7,"London 2026, Sir John Soane's Museum,",,5,,2026-10-11 18:33:48,51.514488,-0.118998,https://live.staticflickr.com/65535/5565843865...,The Island Kings,15775662@N00,photo


### Go through several pages

Now we use a `for` loop to fetch multiple pages. For this introductory example, we deliberately limit the download to **five pages**. Retrieving hundreds of pages can take time and produce many API requests.

The expression `min(total_pages, 5)` ensures we never request more pages than Flickr says exist. Because `range()` excludes its end value, we add `+ 1` to include the final page.

Rather than expanding a DataFrame with every request, we collect the page DataFrames in a list and combine them **once** using `pd.concat()`. The result is stored in `df`, ready for the map in the next section.

In [15]:
pages_to_download = min(total_pages, 5)
all_pages = []

for page in range(1, pages_to_download + 1):
    print(f"Getting page {page} of {total_pages}")
    page_df = get_page(bbox, page, per_page)
    all_pages.append(page_df)

if all_pages:
    df = pd.concat(all_pages, ignore_index=True)
else:
    df = pd.DataFrame(columns=["latitude", "longitude"])

print(f"Photographs retrieved: {len(df)}")
df

Getting page 1 of 38095
Getting page 2 of 38095
Getting page 3 of 38095
Getting page 4 of 38095
Getting page 5 of 38095
Photographs retrieved: 10


,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,55660443740,65535,8c3e08c57e,Exeter Street - 33136,architecture orthographic exeterstreet histori...,26,Charles Octavius Parnell (1864)\n\nCorner deta...,2022-10-24 10:00:00,51.511947,-0.119745,https://live.staticflickr.com/65535/5566044374...,padraic collins,47072881@N06,photo
1,55658638619,65535,761c1ecf62,"London 2026, Sir John Soane's Museum,",,3,,2026-10-11 18:33:56,51.514488,-0.118998,https://live.staticflickr.com/65535/5565863861...,The Island Kings,15775662@N00,photo
2,55658438656,65535,4fe0b610d7,"London 2026, Sir John Soane's Museum,",,5,,2026-10-11 18:33:48,51.514488,-0.118998,https://live.staticflickr.com/65535/5565843865...,The Island Kings,15775662@N00,photo
3,55655080767,65535,393de11e1e,LONDON,,3,LONDON,2026-11-16 05:58:17,51.509837,-0.118778,https://live.staticflickr.com/65535/5565508076...,cyjlrrhc43,205131681@N07,photo
4,55653034107,65535,63600112b1,"London Transport Museum, Covent Garden Piazza,...",tetramesh london england britain greatbritain ...,18,,1988-07-01 00:00:00,51.511994,-0.121654,https://live.staticflickr.com/65535/5565303410...,Tetramesh,94104270@N00,photo
5,55653033867,65535,e4f74d94b0,"Catherine Streer, WC2",tetramesh london england britain greatbritain ...,15,,1988-07-01 00:00:00,51.512682,-0.120409,https://live.staticflickr.com/65535/5565303386...,Tetramesh,94104270@N00,photo
6,55654018021,65535,90cfe68085,"Russell Street, WC2",tetramesh london england britain greatbritain ...,20,,1988-07-01 00:00:00,51.512762,-0.121128,https://live.staticflickr.com/65535/5565401802...,Tetramesh,94104270@N00,photo
7,55654222159,65535,c9bc5ac9d3,"8 Russell Street, WC2",tetramesh london england britain greatbritain ...,27,,1988-07-01 00:00:00,51.512501,-0.121643,https://live.staticflickr.com/65535/5565422215...,Tetramesh,94104270@N00,photo
8,55654017636,65535,85f06af051,"Catherine Street, WC2",tetramesh london england britain greatbritain ...,106,,1988-07-01 00:00:00,51.512682,-0.120409,https://live.staticflickr.com/65535/5565401763...,Tetramesh,94104270@N00,photo
9,55654017481,65535,2a3dae5855,"Market Tavern, 9 Russell Street, WC2",tetramesh london england britain greatbritain ...,23,,1988-07-01 00:00:00,51.512619,-0.121300,https://live.staticflickr.com/65535/5565401748...,Tetramesh,94104270@N00,photo


In [16]:
df.head(10)

,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,55660443740,65535,8c3e08c57e,Exeter Street - 33136,architecture orthographic exeterstreet histori...,26,Charles Octavius Parnell (1864)\n\nCorner deta...,2022-10-24 10:00:00,51.511947,-0.119745,https://live.staticflickr.com/65535/5566044374...,padraic collins,47072881@N06,photo
1,55658638619,65535,761c1ecf62,"London 2026, Sir John Soane's Museum,",,3,,2026-10-11 18:33:56,51.514488,-0.118998,https://live.staticflickr.com/65535/5565863861...,The Island Kings,15775662@N00,photo
2,55658438656,65535,4fe0b610d7,"London 2026, Sir John Soane's Museum,",,5,,2026-10-11 18:33:48,51.514488,-0.118998,https://live.staticflickr.com/65535/5565843865...,The Island Kings,15775662@N00,photo
3,55655080767,65535,393de11e1e,LONDON,,3,LONDON,2026-11-16 05:58:17,51.509837,-0.118778,https://live.staticflickr.com/65535/5565508076...,cyjlrrhc43,205131681@N07,photo
4,55653034107,65535,63600112b1,"London Transport Museum, Covent Garden Piazza,...",tetramesh london england britain greatbritain ...,18,,1988-07-01 00:00:00,51.511994,-0.121654,https://live.staticflickr.com/65535/5565303410...,Tetramesh,94104270@N00,photo
5,55653033867,65535,e4f74d94b0,"Catherine Streer, WC2",tetramesh london england britain greatbritain ...,15,,1988-07-01 00:00:00,51.512682,-0.120409,https://live.staticflickr.com/65535/5565303386...,Tetramesh,94104270@N00,photo
6,55654018021,65535,90cfe68085,"Russell Street, WC2",tetramesh london england britain greatbritain ...,20,,1988-07-01 00:00:00,51.512762,-0.121128,https://live.staticflickr.com/65535/5565401802...,Tetramesh,94104270@N00,photo
7,55654222159,65535,c9bc5ac9d3,"8 Russell Street, WC2",tetramesh london england britain greatbritain ...,27,,1988-07-01 00:00:00,51.512501,-0.121643,https://live.staticflickr.com/65535/5565422215...,Tetramesh,94104270@N00,photo
8,55654017636,65535,85f06af051,"Catherine Street, WC2",tetramesh london england britain greatbritain ...,106,,1988-07-01 00:00:00,51.512682,-0.120409,https://live.staticflickr.com/65535/5565401763...,Tetramesh,94104270@N00,photo
9,55654017481,65535,2a3dae5855,"Market Tavern, 9 Russell Street, WC2",tetramesh london england britain greatbritain ...,23,,1988-07-01 00:00:00,51.512619,-0.121300,https://live.staticflickr.com/65535/5565401748...,Tetramesh,94104270@N00,photo


### Save the results as CSV

We can now export the collected photographs, just as we did in the Pandas tutorial. `to_csv()` creates a conventional comma-separated file. The option `index=False` avoids adding an unnecessary DataFrame index column.

Descriptions sometimes contain line breaks. We replace those with spaces in a separate DataFrame so that text is easier to read in a spreadsheet.

In [17]:
export_df = df.replace(r"\r?\n", " ", regex=True)

export_df.to_csv("london_photos.csv", index=False)

print(f"Saved {len(export_df)} photographs to london_photos.csv")

Saved 10 photographs to london_photos.csv


### Map Display

![Instructional diagram: dataframe to folium](assets/dataframe-to-folium.svg)
Each row in the DataFrame contains the location of one photograph. We convert latitude and longitude into numerical values, remove rows with invalid or out-of-range coordinates from the **mapping copy** (without changing the original DataFrame), and use `zip()` to pair the coordinates for each photograph. Folium expects `[latitude, longitude]`, in that order. 

In [20]:
# Keep the original dataset unchanged
map_df = df.copy()
map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
map_df = map_df.dropna(subset=["latitude", "longitude"])
map_df = map_df[
    map_df["latitude"].between(-90, 90)
    & map_df["longitude"].between(-180, 180)
]

map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    #tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True
)

folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6
).add_to(map_london)

folium.Rectangle(
    bounds=[[lat_south, long_west], [lat_north, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=0.5,
    color="cornflowerblue"
).add_to(map_london)

# zip() pairs one latitude with the longitude from the same row
for latitude, longitude in zip(map_df["latitude"], map_df["longitude"]):
    folium.CircleMarker(
        location=[latitude, longitude],
        radius=6,
        stroke=False,
        fill=True,
        fill_color="orchid",
        fill_opacity=0.3
    ).add_to(map_london)

map_london

**Colours**

The names `orchid` and `cornflowerblue` are [CSS named colours](https://developer.mozilla.org/en-US/docs/Web/CSS/named-color), commonly supported by web-based visualisations such as Folium. You can also use hexadecimal colour values.

---

## Combined Script

This is the combined script: 

### Import libraries and enter credentials

The combined script is a self-contained version of the previous steps. If you run it independently, enter your Flickr key and secret when prompted. As before, `getpass()` keeps them out of the notebook source and outputs.

In [ ]:
from getpass import getpass
import flickrapi
import pandas as pd
import pprint
import folium

api_key = getpass("Flickr API key: ")
api_secret = getpass("Flickr API secret: ")

# Connect to Flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

### Set search parameters

In [ ]:
# A second London example, using a different search area
centre_latitude = 51.51239
centre_longitude = 0.00496

lat_north = 51.51550
lat_south = 51.50941
long_west = 0.00024
long_east = 0.01015

bbox = f"{long_west},{lat_south},{long_east},{lat_north}"
per_page = 250

map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True
)

folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6
).add_to(map_london)

folium.Rectangle(
    bounds=[[lat_south, long_west], [lat_north, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue"
).add_to(map_london)

map_london

### Get amount of pages and photos

In [127]:
photos = flickr.photos.search(bbox=bbox, per_page=per_page, page=1, has_geo=1)

total_pages = photos["photos"]["pages"]
total_photos = photos["photos"]["total"]

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")

Total photos: 3841
Total pages: 16


### Extract information

In [ ]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )
    rows = []
    for photo in response["photos"]["photo"]:
        description = photo.get("description") or {}
        if isinstance(description, dict):
            description = description.get("_content", "")

        rows.append({
            "id": photo.get("id"),
            "server": photo.get("server"),
            "secret": photo.get("secret"),
            "title": photo.get("title", ""),
            "tags": photo.get("tags", ""),
            "views": photo.get("views"),
            "description": description,
            "date_taken": photo.get("datetaken"),
            "latitude": photo.get("latitude"),
            "longitude": photo.get("longitude"),
            "url_s": photo.get("url_s"),
            "owner_name": photo.get("ownername"),
            "owner": photo.get("owner"),
            "media": photo.get("media"),
        })

    columns = [
        "id", "server", "secret", "title", "tags", "views",
        "description", "date_taken", "latitude", "longitude",
        "url_s", "owner_name", "owner", "media",
    ]
    return pd.DataFrame(rows, columns=columns)


pages_to_download = min(int(total_pages), 5)
all_pages = []
for page in range(1, pages_to_download + 1):
    print(f"Getting page {page} of {total_pages}")
    all_pages.append(get_page(bbox, page, per_page))

if all_pages:
    df = pd.concat(all_pages, ignore_index=True)
else:
    df = pd.DataFrame(columns=["latitude", "longitude"])

df.to_csv("london_photos.csv", index=False)
print(f"Photographs retrieved: {len(df)}")
df

### Map display

In [ ]:
# Keep the original dataset unchanged
map_df = df.copy()
map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
map_df = map_df.dropna(subset=["latitude", "longitude"])
map_df = map_df[
    map_df["latitude"].between(-90, 90)
    & map_df["longitude"].between(-180, 180)
]

map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True
)

folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6
).add_to(map_london)

folium.Rectangle(
    bounds=[[lat_south, long_west], [lat_north, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=0.5,
    color="cornflowerblue"
).add_to(map_london)

# zip() pairs one latitude with the longitude from the same row
for latitude, longitude in zip(map_df["latitude"], map_df["longitude"]):
    folium.CircleMarker(
        location=[latitude, longitude],
        radius=1,
        stroke=False,
        fill=True,
        fill_color="orchid",
        fill_opacity=0.3
    ).add_to(map_london)

map_london

## What's Next?  

If you examine this [Flickr Image](https://www.flickr.com/photos/33075566@N08/54187832141), you'll notice additional information associated with the image, such as "Favorites" or "Comments," which were not captured during the initial search.  
To address this, you'll need to iterate through the list and include the missing entries. Refer to the documentation to determine the appropriate API calls required to retrieve this data.

You can also convert the image's capture date into a datetime object, which helps you better analyze and understand the time aspect.